In [11]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_curve, auc, roc_auc_score)
import warnings
warnings.filterwarnings('ignore')

In [25]:
import pandas as pd

In [55]:
df = pd.read_csv('BCDataset.csv')
print("Dataset shape:", df.shape)
print("\nClass distribution:\n", df['Status'].value_counts())


Dataset shape: (4024, 16)

Class distribution:
 Status
Alive    3408
Dead      616
Name: count, dtype: int64


In [ ]:
print("\nMissing values:\n", df.isnull().sum().sum())
print("\nFeatures:\n", df.dtypes)

Preprocessing

In [47]:
le = LabelEncoder()
df_copy = df.copy()
cols = df.select_dtypes(include='object').columns.tolist()
cols.remove('Status')
for col in cols:
    df_copy[col] = le.fit_transform(df_copy[col])
df_copy['Status'] = (df_copy['Status'] == 'Dead').astype(int)


# Set X and y and drop the null
X = df_copy.drop(['Status', 'Unnamed: 3'], axis=1)
y = df_copy['Status']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

In [48]:
print("\nMissing values:\n", df_copy.isnull().sum().sum())


Missing values:
 4024


Model 1: Logistic Regression

In [53]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_s, y_train)

lr_p = lr.predict(X_test_s)
lr_pp = lr.predict_proba(X_test_s)[:,1]
lr_as  = accuracy_score(y_test, lr_p)
lr_cvs  = cross_val_score(lr, X_train_s, y_train, cv=5, scoring='accuracy').mean()
lr_auc  = roc_auc_score(y_test, lr_pp)
print(f"Logistic Regression:\nAccuracy: {lr_as:.4f}\nCV: {lr_cvs:.4f}\nAUC: {lr_auc:.4f}")
print(classification_report(y_test, lr_p, target_names=['Alive','Dead']))

Logistic Regression:
Accuracy: 0.9006
CV: 0.8907
AUC: 0.8545
              precision    recall  f1-score   support

       Alive       0.91      0.98      0.94       682
        Dead       0.81      0.46      0.58       123

    accuracy                           0.90       805
   macro avg       0.86      0.72      0.76       805
weighted avg       0.89      0.90      0.89       805



Model 2: Random Forest

In [54]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

rf_p = rf.predict(X_test)
rf_pp = rf.predict_proba(X_test)[:,1]
rf_as  = accuracy_score(y_test, rf_p)
rf_cvs   = cross_val_score(rf, X_train, y_train, cv=5, scoring='accuracy').mean()
rf_auc  = roc_auc_score(y_test, rf_pp)
print(f"Random Forest:\nAcc: {rf_as:.4f}\nCV: {rf_cvs:.4f}\nAUC: {rf_auc:.4f}")
print(classification_report(y_test, rf_p, target_names=['Alive','Dead']))

Random Forest:
Acc: 0.9118
CV: 0.9006
AUC: 0.8264
              precision    recall  f1-score   support

       Alive       0.92      0.99      0.95       682
        Dead       0.87      0.50      0.63       123

    accuracy                           0.91       805
   macro avg       0.89      0.74      0.79       805
weighted avg       0.91      0.91      0.90       805

